# Lab Stage 1: Building Your First Research Workflow

## Single-Subject EEG Emotion Analysis

**Workflow:** Obtaining data ➡️ Analyzing data ➡️ Visualizing results ➡️ Documenting research

**Platform:** Google Colab (standard CPU runtime)
**Domain:** Brain and behavioral research, emotion and EEG
**Dataset:** EEG recorded while one participant (S01) played four emotionally different computer games: boring, calm, horror and funny

### How this lab works
You are the **Workflow Designer**. For each module you copy an **AI Prompt Challenge** into your AI assistant (Claude, ChatGPT, GitHub Copilot or similar), check what it gives back, paste the code into the cell marked `TODO`, and run it. You don't need to write code yourself, but you do need to judge whether the code does what you asked.

When all four modules are connected, use the **Embedded Playground** at the end to test your workflow under different analysis choices.

## 🛡️ Before You Start: Check What the AI Gives You

AI-written code is fast, but it isn't automatically correct or safe. A 2026 study that tested six leading AI models found that **25.1% of the code samples they generated contained a confirmed, exploitable security vulnerability**. Treat AI code the way you would treat a stranger's lab protocol: read it before you run it.

Every time you paste AI-generated code into this notebook, run this three-question check:

1. **Does it do what I asked?** Compare what it prints (row counts, column names, the number of results) with the expected result given after each step.
2. **Does it do anything I didn't ask for?** Be suspicious of code that installs unfamiliar packages, downloads from or uploads to websites you didn't name, deletes files, or asks for passwords or keys.
3. **Can I explain each step in plain words?** If not, ask the AI: *"Explain this code line by line for a biologist who doesn't program."*

## Module 1: Obtaining Data

The [dataset](https://www.kaggle.com/datasets/wajahat1064/emotion-recognition-using-eeg-and-computer-games) contains EEG recorded from 14 electrodes (AF3 to T8) at 128 samples per second while participants played four computer games for about 5 minutes each. We will analyze participant S01.

#### 🤖 AI Prompt Challenge

Copy the code cell below **and** this text into your AI assistant:

> "Write a function `get_data(data_config)` that takes the `DATA_CONFIG` dictionary below, reads each file with pandas using the URLs in the values, drops any column whose name starts with 'Unnamed' (each file has an empty extra column at the end), adds a column called `game` holding the dictionary key, concatenates all the dataframes into one, and returns it."

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats, signal
from datetime import date

DATA_CONFIG = {
    "boring": "https://github.com/cyneuro/CI-BioEng-Class/raw/refs/heads/main/emotion_recognition/data/S01G1AllChannels.csv",
    "calm": "https://github.com/cyneuro/CI-BioEng-Class/raw/refs/heads/main/emotion_recognition/data/S01G2AllChannels.csv",
    "horror": "https://github.com/cyneuro/CI-BioEng-Class/raw/refs/heads/main/emotion_recognition/data/S01G3AllChannels.csv",
    "funny": "https://github.com/cyneuro/CI-BioEng-Class/raw/refs/heads/main/emotion_recognition/data/S01G4AllChannels.csv",
}
FS = 128  # samples per second

In [ ]:
# TODO: Paste your AI-generated code here, then run this cell.

In [ ]:
data = get_data(DATA_CONFIG)
print(data.shape)
data.head()

✅ **Expected result:** `(153008, 15)`, that is 4 games × 38,252 samples, with 14 electrode columns plus `game`. If you see 16 columns, the empty `Unnamed: 14` column is still there: ask the AI to drop it.

## Module 2: Analyzing Data

Our question: **does brain activity differ across the four games?**

### Why not just compare the raw voltage?
The obvious first try is to compare the average voltage at each electrode across games. Run the cell below to see why that doesn't work here.

In [ ]:
# Pre-built: average voltage per game at three electrodes
data.groupby("game")[["AF3", "F4", "O1"]].mean().round(3)

The averages are all close to **zero** for every game. These recordings were preprocessed so that each channel is centered on zero, so comparing averages can't find a difference, even if the brain was doing something quite different in each game. (An earlier version of this lab did exactly that and found nothing on any electrode.)

What does change is the **rhythm** of the signal. EEG contains oscillations at different speeds, and their strength rises and falls with alertness and emotion. We'll use the **alpha rhythm (8–13 cycles per second)**, which is strongest when a person is relaxed and weakens when they are engaged or alert.

### Step 2a: Measure alpha power in 30-second blocks
We'll cut each 5-minute recording into 30-second blocks and measure alpha power in each block. That gives 9 blocks per game, so 9 measurements per game for each electrode, which is what the statistics need.

#### 🤖 AI Prompt Challenge
> "My dataframe `data` has 14 EEG electrode columns (AF3 to T8) sampled at 128 Hz, plus a `game` column. Write a function `alpha_power_blocks(data, band=(8, 13), block_seconds=30, fs=128)` that, separately for each game (keep the games in the order they appear), cuts the recording into consecutive non-overlapping blocks of `block_seconds` (drop any leftover samples at the end). For each block and each electrode, compute the power spectrum with `scipy.signal.periodogram` using a Hann window, average the power between the two band edges (including the lower edge, excluding the upper edge), and take log10. Return a dataframe with one row per block and columns `game`, `block` (numbered from 1) and one column per electrode."

In [ ]:
# TODO: Paste your AI-generated code here, then run this cell.

In [ ]:
power = alpha_power_blocks(data)
print(power.shape)
print(power.groupby("game").size())
power.head()

✅ **Expected result:** `(36, 16)`, that is 4 games × 9 blocks, with columns `game`, `block` and the 14 electrodes.

### Step 2b: Why testing 14 electrodes needs a correction
We're about to run one test per electrode: 14 tests. With the usual cut-off (p < 0.05), each test has a 5% chance of a **false positive**: calling a difference "significant" when there's nothing there. Run 14 tests and those chances add up.

The pre-built cell below shows this with **pure random noise**, where nothing real is happening, repeated 2,000 times. It compares no correction with two standard corrections:
- **Bonferroni:** require p < 0.05 ÷ 14 for every test. Simple and strict.
- **False discovery rate (FDR, Benjamini–Hochberg):** a less strict method that keeps the share of false findings among your "significant" results at about 5%. It's the one we'll use.

(Takes about 15 seconds.)

In [ ]:
# Pre-built: 14 "electrodes" of pure noise, 4 "games", repeated 2,000 times
def bh_reject(p_values, alpha=0.05):
    p = np.asarray(p_values)
    n = len(p)
    order = np.argsort(p)
    below = p[order] <= np.arange(1, n + 1) / n * alpha
    if not below.any():
        return np.zeros(n, dtype=bool)
    return p <= p[order][np.max(np.where(below)[0])]

rng = np.random.default_rng(0)
runs = 2000
any_uncorrected = any_bonferroni = any_fdr = 0
for _ in range(runs):
    p = np.array([stats.f_oneway(*[rng.normal(size=9) for _ in range(4)]).pvalue for _ in range(14)])
    any_uncorrected += np.any(p < 0.05)
    any_bonferroni += np.any(p < 0.05 / 14)
    any_fdr += bh_reject(p).any()

print("Runs where at least one electrode looked 'significant' although nothing was real:")
print(f"  no correction: {100 * any_uncorrected / runs:.0f}%")
print(f"  Bonferroni:    {100 * any_bonferroni / runs:.0f}%")
print(f"  FDR:           {100 * any_fdr / runs:.0f}%")

Without correction, about **half** of the runs report at least one "significant" electrode in data that is pure noise. With either correction it drops to about 5–6%, close to the 5% we intended. That is why the analysis below reports both counts.

### Step 2c: Test each electrode, with correction and effect size
A p-value tells you how surprising a difference would be if nothing were going on. It doesn't tell you how **big** the difference is. So we'll also report an **effect size**, eta-squared (η²): the share of the variation in alpha power that is explained by which game was being played (0 = none, 1 = all of it).

#### 🤖 AI Prompt Challenge
> "My dataframe `power` has columns `game`, `block` and one column per EEG electrode. Write a function `run_anova(power, show_table=True)` that, for each electrode separately, runs a one-way ANOVA (`scipy.stats.f_oneway`) comparing the values across games, and computes eta-squared (between-group sum of squares divided by total sum of squares). Then apply a Benjamini–Hochberg false discovery rate correction at 0.05 across all the electrodes' p-values; write the correction yourself with numpy rather than importing statsmodels. Return a dataframe with columns `electrode`, `F`, `p_value`, `eta_squared`, `significant_uncorrected` (p < 0.05) and `significant_fdr`. If `show_table` is True, print the table, then print how many electrodes are significant before and after correction."

In [ ]:
# TODO: Paste your AI-generated code here, then run this cell.

In [ ]:
anova_result = run_anova(power)

✅ **Expected result:** most electrodes differ across games, both before and after correction, with η² mostly between about 0.3 and 0.6, meaning the game explains roughly a third to over half of the block-to-block variation in alpha power.

💡 **Notice:** here the correction barely changes anything, because most differences are large. Correction matters most when results are **borderline**, and you'll see a case like that in Playground Task 1.

## Module 3: Visualizing Results

#### 🤖 AI Prompt Challenge
> "Write a function `plot_results(power, anova_result)` that draws a 4 × 4 grid of seaborn bar plots, one panel per electrode, showing the mean log alpha power for each game with standard-error bars (`errorbar='se'`). Title each panel with the electrode name followed by `*` if `significant_fdr` is True in `anova_result` for that electrode, or `NS` if not. Remove the two unused panels and call `plt.tight_layout()`."

In [ ]:
# TODO: Paste your AI-generated code here, then run this cell.

In [ ]:
plot_results(power, anova_result)

✅ **Expected result:** 14 panels, each titled `*` or `NS`. Look at which games tend to have the **highest** alpha power, then think back to what alpha means (relaxed vs. engaged).

## Module 4: Documenting Research

A good workflow records **what was run and with which settings**, so anyone, including you next month, can reproduce or compare it.

#### 🤖 AI Prompt Challenge
Copy the code cell below **and** this text into your AI assistant:
> "Write a function `document_research(anova_result, settings, log_file='lab1_run_log.csv')` that builds a one-row pandas dataframe with: today's date, every key and value in the `settings` dictionary, the number of electrodes tested, the number significant before correction (`significant_uncorrected`) and the number significant after FDR correction (`significant_fdr`). Append the row to `log_file` (create the file with a header if it doesn't exist yet), print the row, and return the full log as a dataframe."

In [ ]:
SETTINGS = {"games": "all four", "band_hz": "8-13 (alpha)", "block_seconds": 30}

In [ ]:
# TODO: Paste your AI-generated code here, then run this cell.

In [ ]:
log = document_research(anova_result, SETTINGS)

## 🧪 Embedded Playground

Use AI to adapt your workflow. Log every run with `document_research()` so you can compare them.

**Task 1. Two games only.** Which two games do you expect to produce the most different emotional responses? A common guess is **calm vs. horror**. Re-run Modules 2–4 using only those two games (filter `data` to those games, then rebuild `power`). How many electrodes are significant before and after correction?

**Task 2. A different analysis choice.** Try a different rhythm, **theta (4–8 Hz)** or **beta (13–30 Hz)**, or a different block length (**10 seconds**). Do the counts change? What does that tell you about how analysis choices affect conclusions?

**Task 3. Is it really the game?** Take a single game and compare its **first half** of blocks with its **second half**, as if they were two conditions. Nothing changed between the halves except time. Do any electrodes come out "significant"?

In [ ]:
# Task 1: TODO, paste your AI-generated code here.

In [ ]:
# Task 2: TODO, paste your AI-generated code here.

In [ ]:
# Task 3: TODO, paste your AI-generated code here.

## ✍️ Reflection

1. **What did you observe?** How many electrodes differed across the four games, before and after correction? Which games had the highest alpha power, and does that fit what alpha is thought to reflect?
2. **When did correction matter?** Compare the main analysis with Playground Task 1. Why did correction change the answer in one case but not the other?
3. **Stage independence.** `run_anova()` only receives the `power` table. It doesn't know whether that table came from alpha, theta, 30-second or 10-second blocks. Why is that useful?
4. **Checking the AI.** Did any AI-generated code need fixing? What did the three-question check catch?

## ⚠️ What These Results Can and Can't Tell Us
This lab analyzed **one person**, with **one recording per game**. The differences are real differences between those four recordings, but they can't prove the *game* caused them: the person may simply have become more tired or more alert over the session (Playground Task 3). To make a claim about emotions in general, we need many participants. That's exactly what **Lab Stage 2** does, with all 28 participants on FABRIC.